## Setup and MLflow Initialization
In this section I initialize the environment and connect the project to DagsHub and MLflow. This allows us to track every experiment remotely, ensuring that our model parameters and metrics are saved in a professional cloud registry.

In [ ]:
import dagshub
import mlflow
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_squared_log_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor, AdaBoostRegressor

# Set Up Tracking Environment and Remote Server
os.environ['MLFLOW_TRACKING_USERNAME'] = 'GigiSichinava'
dagshub.init(repo_owner='GigiSichinava', repo_name='ML')
mlflow.set_tracking_uri("https://dagshub.com/GigiSichinava/ML.mlflow")
mlflow.set_experiment("House_Price_Regression")


## Cleaning and Feature Engineering
Here I load the Kaggle dataset and focus on handling missing values (NaNs) and identifying numeric columns. By filling missing values with zeros I create a stable baseline for our regression models.

In [ ]:
# Data Loading and Preparation
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# Select only numeric columns and drop 'ID'
numeric_cols = train_df.select_dtypes(include=[np.number]).columns


## Feature Selection
For this experiment I filtered for numeric features and removed non-essential columns like 'Id'. I then split the data into 80% training and 20% validation sets to ensure we can properly test the model's performance on unseen data.

In [ ]:
# Filtering for numeric columns only and removing IDs
X = train_df[numeric_cols].drop(['SalePrice', 'Id'], axis=1, errors='ignore').fillna(0)
y = train_df['SalePrice']

# Split into 80/20 portion
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Log Data Setup to DagsHub
with mlflow.start_run(run_name="Final_Data_Prep"):
    mlflow.log_param("train_shape", str(X_train.shape))
    mlflow.log_param("val_shape", str(X_val.shape))
    print(f"Setup Complete: Training on {X_train.shape[0]} houses.")


## Model Training and MLflow Tracking
I created a automated function that trains different models (Linear Regression, XGBoost and Others), logs their performance metrics (RMSLE, RMSE, R2) to MLflow and pushes each model to the DagsHub Model Registry. This allows for seamless version control and ensures the best performing model can be easily retrieved for inference.

In [ ]:
# The Metric Machine Function
def train_and_log(model, model_name):
    with mlflow.start_run(run_name=model_name):
        model.fit(X_train, y_train)
        preds = model.predict(X_val)

        # Metrics
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        mae = mean_absolute_error(y_val, preds)
        r2 = r2_score(y_val, preds)
        rmsle = np.sqrt(mean_squared_log_error(y_val, np.clip(preds, 0, None)))

        # Log Metrics
        mlflow.log_param("model_type", model_name)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2_score", r2)
        mlflow.log_metric("rmsle", rmsle)

        # Save to Model Registry
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            registered_model_name=model_name
        )

        # Printout
        print(f"Model logged: {model_name}")
        print(f"   RMSLE:          {rmsle:.4f}")
        print(f"   RMSE:           ${rmse:,.0f}")
        print(f"   MAE:            ${mae:,.0f}")
        print(f"   R2 Score:       {r2:.4f}")
        print("-" * 30)

        return model

##### Linear Regression

In [ ]:
lr_model = train_and_log(LinearRegression(), "Linear_Regression")

##### Decision Tree

In [ ]:
dt_model = train_and_log(DecisionTreeRegressor(random_state=42), "Decision_Tree")

##### Random Forest

In [ ]:
rf_model = train_and_log(RandomForestRegressor(n_estimators=100, random_state=42), "Random_Forest")

##### Tuned Random Forest

In [ ]:
rf_tuned = train_and_log(RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42), "Tuned_Random_Forest")

#### XGBoost
After comparing the results, XGBoost emerged as the "Champion" model with the lowest RMSLE (0.1434). This model was pushed to the production registry for the final inference task.

In [ ]:
xgb_model = train_and_log(xgb.XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=5), "XGBoost")

##### Ridge Regression

In [ ]:
ridge_model = train_and_log(Ridge(alpha=1.0), "Ridge_Regression")

##### Gradient Boosting

In [ ]:
gbr_model = train_and_log(GradientBoostingRegressor(n_estimators=100, random_state=42), "Gradient_Boosting")

##### AdaBoost

In [ ]:
ada_model = train_and_log(AdaBoostRegressor(n_estimators=100, random_state=42), "AdaBoost")